<a href="https://colab.research.google.com/github/izzat-ai/learning-ai/blob/main/numpy/agricultural_monitoring.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [17]:
import numpy as np

In [18]:
satellite_data = np.array([
    [120, 150, 110, 450], # Region 1: Dense forest
    [180, 130, 120, 210], # Region 2: Dry land/Building
    [100, 200, 90,  500], # Region 3: Cropland
    [255, 255, 255, 10],  # Region 4: Cloud (Error data)
    [140, 160, 130, 400]  # Region 5: Orchard/Garden
], dtype=float)

# Region areas (in square meters)
area_sq_m = np.array([5000, 2000, 7500, 1000, 3000])

satellite_data

array([[120., 150., 110., 450.],
       [180., 130., 120., 210.],
       [100., 200.,  90., 500.],
       [255., 255., 255.,  10.],
       [140., 160., 130., 400.]])

In [19]:
# red, green, blue, near-infrared
satellite_data.shape

(5, 4)

In [20]:
# The NIR channel (column 4) should always be greater than the Red channel (column 1)
np.argmax(satellite_data[:, -1] < satellite_data[:, 0])

np.int64(3)

In [21]:
# delete the Red data that is greater than the NIR channel.
satellite_data = np.delete(satellite_data, 3, axis=0)

area_sq_m = np.delete(area_sq_m, 3)

In [22]:
satellite_data

array([[120., 150., 110., 450.],
       [180., 130., 120., 210.],
       [100., 200.,  90., 500.],
       [140., 160., 130., 400.]])

In [23]:
area_sq_m

array([5000, 2000, 7500, 3000])

In [24]:
# Vegetation Index Calculation
ndvi = (satellite_data[:, -1] - satellite_data[:, 0]) / (satellite_data[:, -1] + satellite_data[:, 0])
ndvi

array([0.5789, 0.0769, 0.6667, 0.4815])

In [25]:
# add the detected NDVI to the main array
np.set_printoptions(suppress=True, precision=4)
satellite_data = np.append(satellite_data, ndvi.reshape(-1, 1), axis=1)
satellite_data

array([[120.    , 150.    , 110.    , 450.    ,   0.5789],
       [180.    , 130.    , 120.    , 210.    ,   0.0769],
       [100.    , 200.    ,  90.    , 500.    ,   0.6667],
       [140.    , 160.    , 130.    , 400.    ,   0.4815]])

In [26]:
condlist = [
    ndvi > 0.5,
    (ndvi > 0.2) & (ndvi <= 0.5),
    ndvi <= 0.2
]

# 'dense plant'-1.0, 'sparse plant'-0.5, 'no plant'-0.0
choicelist = [1.0, 0.5, 0.0]

classification_score = np.select(condlist, choicelist)
classification_score

array([1. , 0. , 1. , 0.5])

In [27]:
# append the classification answer to the main array
satellite_data = np.append(satellite_data, classification_score.reshape(-1, 1), axis=1)
satellite_data

array([[120.    , 150.    , 110.    , 450.    ,   0.5789,   1.    ],
       [180.    , 130.    , 120.    , 210.    ,   0.0769,   0.    ],
       [100.    , 200.    ,  90.    , 500.    ,   0.6667,   1.    ],
       [140.    , 160.    , 130.    , 400.    ,   0.4815,   0.5   ]])

In [28]:
# green zone size
green_zone_sizes = satellite_data[:, -1] * area_sq_m
green_zone_sizes

array([5000.,    0., 7500., 1500.])

In [29]:
# All data from the area with the highest NDVI
satellite_data[satellite_data[:, -2].argmax()]

array([100.    , 200.    ,  90.    , 500.    ,   0.6667,   1.    ])

In [30]:
# Determining the average NDVI of all regions
satellite_data[:, -2].mean()

satellite_data

array([[120.    , 150.    , 110.    , 450.    ,   0.5789,   1.    ],
       [180.    , 130.    , 120.    , 210.    ,   0.0769,   0.    ],
       [100.    , 200.    ,  90.    , 500.    ,   0.6667,   1.    ],
       [140.    , 160.    , 130.    , 400.    ,   0.4815,   0.5   ]])

In [32]:
transposed_satellite = satellite_data.T
transposed_satellite

array([[120.    , 180.    , 100.    , 140.    ],
       [150.    , 130.    , 200.    , 160.    ],
       [110.    , 120.    ,  90.    , 130.    ],
       [450.    , 210.    , 500.    , 400.    ],
       [  0.5789,   0.0769,   0.6667,   0.4815],
       [  1.    ,   0.    ,   1.    ,   0.5   ]])

In [33]:
# calculating average values for each channel
# Note: Fixed the axis logic here to calculate column means correctly after transpose
np.mean(transposed_satellite, axis=1)

array([135.   , 160.   , 112.5  , 390.   ,   0.451,   0.625])

####**Conclusion**

- `Area 4` (Cloud Cover): This area was obscured by clouds, causing the satellite to err and send incorrect numbers. We have removed this area from the list entirely to ensure our calculations are accurate.
- Best Land (`Area 3` - Cropland): This land has the highest NDVI (0.66). This means that the plants here are very dense, healthy, and the crops are growing well.
- Worst Land (`Area 2` - Dry Land): This land has a score of close to zero (0.07). This means there is almost no vegetation here. This is either a construction site or a dry, waterless land that needs to be looked at by experts.
- It was determined that there is a total of 13,500 sq.m. of real green space in the region
- The park in area 5 is currently in average condition (semi-green). If it is well maintained, the size of the green area can be increased by another 1,500 sq m